# Notebook 07 — Incertidumbre y clasificación selectiva

## Grain AI–MDPI

Este notebook analiza si los modelos pueden reducir el riesgo de clasificación absteniéndose de emitir una decisión en muestras de baja confianza.

### Objetivos

1. Mantener como modelos principales los campeones seleccionados exclusivamente mediante validación cruzada en entrenamiento:
   - Dry Bean: `R1_full_native + rbf_svm`.
   - INIAP Rice: `R4_hybrid_log_area + rbf_svm`.
2. Calcular tres puntuaciones de certeza:
   - Probabilidad máxima.
   - Margen entre las dos probabilidades principales.
   - Uno menos la entropía normalizada.
3. Seleccionar la puntuación de certeza mediante la partición de calibración.
4. Fijar umbrales en calibración para coberturas objetivo de 100%, 95%, 90%, 80% y 70%.
5. Aplicar esos umbrales sin cambios al conjunto de prueba.
6. Estimar intervalos de confianza mediante bootstrap estratificado por clase y agrupado por `duplicate_group_id`.
7. Generar curvas riesgo–cobertura, AURC, análisis por clase y reducción del riesgo.
8. Comparar probabilidades nativas y escaladas por temperatura como análisis de sensibilidad.

### Restricciones metodológicas

- No se entrenan modelos.
- No se reajustan hiperparámetros.
- No se elige un nuevo campeón con la prueba.
- Los umbrales de rechazo se fijan únicamente en calibración.
- La prueba se utiliza una sola vez para evaluación.
- El análisis se limita a incertidumbre predictiva en datos morfométricos tabulares.
- No se formulan afirmaciones sobre iluminación, cámaras o segmentación.

In [ ]:
# ============================================================
# 1. Dependencias y entorno
# ============================================================
import importlib
import subprocess
import sys

REQUIRED_PACKAGES = {
    "pyarrow": "pyarrow",
}

for module_name, package_name in REQUIRED_PACKAGES.items():
    if importlib.util.find_spec(module_name) is None:
        subprocess.check_call(
            [sys.executable, "-m", "pip", "install", "-q", package_name]
        )

from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json
import math
import os
import platform
import shutil
import zipfile
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from IPython.display import display
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    matthews_corrcoef,
)
from statsmodels.stats.multitest import multipletests

warnings.filterwarnings("ignore")

print("Python:", sys.version.split()[0])
print("Plataforma:", platform.platform())
print("CPU disponibles:", os.cpu_count())

## Configuración reproducible

Para resultados destinados al manuscrito se recomienda conservar:

```python
N_BOOTSTRAP = 2000
```

Las coberturas son metas fijadas en calibración. La cobertura obtenida en prueba puede diferir porque no se recalibra el umbral con la prueba.

In [ ]:
# ============================================================
# 2. Configuración
# ============================================================
RANDOM_SEED = 20260804
N_BOOTSTRAP = 2000
CONFIDENCE_LEVEL = 0.95

DATASETS = ["dry_bean", "iniap_rice"]
PROBABILITY_STATUSES = ["native", "temperature_scaled"]
CERTAINTY_SCORES = [
    "max_probability",
    "top2_margin",
    "one_minus_normalized_entropy",
]
TARGET_COVERAGES = [1.00, 0.95, 0.90, 0.80, 0.70]
PRIMARY_PROBABILITY_STATUS = "temperature_scaled"
PRIMARY_CLASS_ANALYSIS_COVERAGE = 0.90

LOCAL_ROOT = Path("/content/Grain_AI_MDPI_notebook07")
INPUT_DIR = LOCAL_ROOT / "00_input"
EXTRACT04_DIR = LOCAL_ROOT / "01_extracted_notebook04"
EXTRACT05_DIR = LOCAL_ROOT / "02_extracted_notebook05"
EXTRACT06_DIR = LOCAL_ROOT / "03_extracted_notebook06"
OUTPUT_ROOT = LOCAL_ROOT / "04_outputs"

PROTOCOL_DIR = OUTPUT_ROOT / "03_protocol"
RESULTS_DIR = OUTPUT_ROOT / "07_results" / "07_SELECTIVE_UNCERTAINTY"
FIGURES_DIR = OUTPUT_ROOT / "08_figures" / "07_SELECTIVE_UNCERTAINTY"
TABLES_DIR = OUTPUT_ROOT / "09_tables" / "07_SELECTIVE_UNCERTAINTY"

for directory in [
    INPUT_DIR,
    EXTRACT04_DIR,
    EXTRACT05_DIR,
    EXTRACT06_DIR,
    PROTOCOL_DIR,
    RESULTS_DIR,
    FIGURES_DIR,
    TABLES_DIR,
]:
    directory.mkdir(parents=True, exist_ok=True)

print("Réplicas bootstrap:", N_BOOTSTRAP)
print("Coberturas objetivo:", TARGET_COVERAGES)
print("Semilla:", RANDOM_SEED)

In [ ]:
# ============================================================
# 3. Montaje de Drive y localización de entradas
# ============================================================
try:
    from google.colab import drive, files
    drive.mount("/content/drive", force_remount=False)
    IN_COLAB = True
except Exception:
    IN_COLAB = False
    files = None
    print("No se detectó Google Colab. Se usarán rutas locales.")

DRIVE_PROJECT_ROOT = Path("/content/drive/MyDrive/Grain_Robustness_Q1")
DRIVE_PROJECT_ROOT.mkdir(parents=True, exist_ok=True)

ZIP_FILENAMES = {
    "notebook04": "NOTEBOOK04_MODEL_SELECTION_OUTPUTS.zip",
    "notebook05": "NOTEBOOK05_BASELINE_CALIBRATION_OUTPUTS.zip",
    "notebook06": "NOTEBOOK06_PAIRED_INFERENCE_OUTPUTS.zip",
}

def find_exact_file(root: Path, filename: str):
    if not root.exists():
        return None
    matches = sorted(root.rglob(filename))
    if not matches:
        return None
    return max(matches, key=lambda path: path.stat().st_mtime)

def obtain_input_zip(filename: str) -> Path:
    found = find_exact_file(DRIVE_PROJECT_ROOT, filename)
    if found is not None:
        print("Encontrado en Drive:", found)
        return found

    local_candidate = Path("/content") / filename
    if local_candidate.exists():
        print("Encontrado en /content:", local_candidate)
        return local_candidate

    if IN_COLAB and files is not None:
        print(f"No se encontró {filename}. Selecciónelo para subirlo.")
        uploaded = files.upload()
        if filename not in uploaded:
            raise FileNotFoundError(
                f"Debe subir exactamente el archivo {filename}."
            )
        destination = INPUT_DIR / filename
        destination.write_bytes(uploaded[filename])
        return destination

    raise FileNotFoundError(
        f"No se encontró {filename} en {DRIVE_PROJECT_ROOT}."
    )

zip_paths = {
    key: obtain_input_zip(filename)
    for key, filename in ZIP_FILENAMES.items()
}

display(
    pd.DataFrame(
        [
            {"source": key, "path": str(path)}
            for key, path in zip_paths.items()
        ]
    )
)

In [ ]:
# ============================================================
# 4. Extracción limpia y archivos requeridos
# ============================================================
def clean_extract(zip_path: Path, destination: Path):
    if destination.exists():
        shutil.rmtree(destination)
    destination.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(zip_path, "r") as archive:
        archive.extractall(destination)

clean_extract(zip_paths["notebook04"], EXTRACT04_DIR)
clean_extract(zip_paths["notebook05"], EXTRACT05_DIR)
clean_extract(zip_paths["notebook06"], EXTRACT06_DIR)

SELECTED_CONFIG_FILE = (
    EXTRACT04_DIR
    / "07_results"
    / "04_MODEL_SELECTION"
    / "03_selected_model_configurations.csv"
)
PREDICTION_ROOT = (
    EXTRACT05_DIR
    / "06_predictions"
    / "05_BASELINE"
)
NOTEBOOK06_VALIDATION_FILE = (
    EXTRACT06_DIR
    / "07_results"
    / "06_PAIRED_INFERENCE"
    / "02_final_inference_validation.csv"
)
NOTEBOOK06_CHAMPIONS_FILE = (
    EXTRACT06_DIR
    / "09_tables"
    / "06_PAIRED_INFERENCE"
    / "Table_training_selected_champions_with_test_ci.csv"
)

required_paths = [
    SELECTED_CONFIG_FILE,
    PREDICTION_ROOT,
    NOTEBOOK06_VALIDATION_FILE,
    NOTEBOOK06_CHAMPIONS_FILE,
]
missing = [str(path) for path in required_paths if not path.exists()]
if missing:
    raise FileNotFoundError(
        "Faltan archivos requeridos:\n" + "\n".join(missing)
    )

selected_configurations = pd.read_csv(SELECTED_CONFIG_FILE)
notebook06_validation = pd.read_csv(NOTEBOOK06_VALIDATION_FILE)
notebook06_champions = pd.read_csv(NOTEBOOK06_CHAMPIONS_FILE)

if not notebook06_validation["passed"].astype(bool).all():
    display(notebook06_validation)
    raise RuntimeError(
        "El Notebook 06 no superó todas sus validaciones."
    )

print("Notebook 06 validado correctamente.")
display(notebook06_champions)

In [ ]:
# ============================================================
# 5. Confirmación de campeones seleccionados en entrenamiento
# ============================================================
training_champions = (
    selected_configurations.sort_values(
        ["dataset_id", "macro_f1_mean", "log_loss_mean", "candidate_id"],
        ascending=[True, False, True, True],
    )
    .groupby("dataset_id", as_index=False)
    .head(1)
    .reset_index(drop=True)
)

champion_columns = [
    "dataset_id",
    "representation",
    "algorithm",
    "candidate_id",
]

champion_crosscheck = training_champions[
    champion_columns
].merge(
    notebook06_champions[champion_columns],
    on="dataset_id",
    how="outer",
    suffixes=("_notebook04", "_notebook06"),
)

champion_crosscheck["matches"] = (
    (
        champion_crosscheck["representation_notebook04"]
        == champion_crosscheck["representation_notebook06"]
    )
    & (
        champion_crosscheck["algorithm_notebook04"]
        == champion_crosscheck["algorithm_notebook06"]
    )
    & (
        champion_crosscheck["candidate_id_notebook04"]
        == champion_crosscheck["candidate_id_notebook06"]
    )
)

if not champion_crosscheck["matches"].all():
    display(champion_crosscheck)
    raise RuntimeError(
        "Los campeones de los notebooks 04 y 06 no coinciden."
    )

display(champion_crosscheck)

In [ ]:
# ============================================================
# 6. Carga de predicciones de calibración y prueba
# ============================================================
def load_class_map(path: Path):
    mapping = json.loads(path.read_text(encoding="utf-8"))
    return [str(mapping[str(index)]) for index in range(len(mapping))]

def load_prediction(
    dataset_id,
    representation,
    algorithm,
    split,
):
    directory = PREDICTION_ROOT / dataset_id / representation
    prediction_path = (
        directory / f"{algorithm}_{split}_predictions.parquet"
    )
    class_map_path = directory / f"{algorithm}_class_map.json"

    if not prediction_path.exists():
        raise FileNotFoundError(prediction_path)
    if not class_map_path.exists():
        raise FileNotFoundError(class_map_path)

    frame = pd.read_parquet(prediction_path)
    class_order = load_class_map(class_map_path)

    required_columns = {
        "record_id",
        "duplicate_group_id",
        "class_label",
        "partition",
        "predicted_native",
        "predicted_calibrated",
        "correct_native",
        "correct_calibrated",
    }
    missing_columns = sorted(required_columns - set(frame.columns))
    if missing_columns:
        raise ValueError(
            f"Columnas faltantes en {prediction_path}: {missing_columns}"
        )

    native_columns = [
        f"prob_native_{index}"
        for index in range(len(class_order))
    ]
    calibrated_columns = [
        f"prob_calibrated_{index}"
        for index in range(len(class_order))
    ]

    for column in native_columns + calibrated_columns:
        if column not in frame.columns:
            raise ValueError(
                f"Falta {column} en {prediction_path}"
            )

    frame = frame.copy()
    frame["record_id"] = frame["record_id"].astype(str)
    frame["duplicate_group_id"] = (
        frame["duplicate_group_id"].astype(str)
    )
    frame["class_label"] = frame["class_label"].astype(str)
    frame = frame.sort_values("record_id").reset_index(drop=True)

    expected_partition = (
        "calibration" if split == "calibration" else "test"
    )
    if not (frame["partition"] == expected_partition).all():
        raise ValueError(
            f"{prediction_path} contiene particiones inesperadas."
        )

    for columns in [native_columns, calibrated_columns]:
        probabilities = frame[columns].to_numpy(dtype=float)
        if not np.isfinite(probabilities).all():
            raise ValueError(
                f"Probabilidades no finitas: {prediction_path}"
            )
        if not np.allclose(
            probabilities.sum(axis=1),
            1.0,
            atol=1e-8,
        ):
            raise ValueError(
                f"Probabilidades no normalizadas: {prediction_path}"
            )

    return {
        "frame": frame,
        "class_order": class_order,
        "native_probability_columns": native_columns,
        "calibrated_probability_columns": calibrated_columns,
        "prediction_path": prediction_path,
    }

predictions = {}

for _, champion in training_champions.iterrows():
    dataset_id = champion["dataset_id"]
    representation = champion["representation"]
    algorithm = champion["algorithm"]

    for split in ["calibration", "test"]:
        key = (
            dataset_id,
            representation,
            algorithm,
            split,
        )
        predictions[key] = load_prediction(*key)

print("Matrices de predicción cargadas:", len(predictions))

In [ ]:
# ============================================================
# 7. Auditoría de alineación y separación de grupos
# ============================================================
prediction_audit_rows = []

for _, champion in training_champions.iterrows():
    dataset_id = champion["dataset_id"]
    representation = champion["representation"]
    algorithm = champion["algorithm"]

    calibration_bundle = predictions[
        (dataset_id, representation, algorithm, "calibration")
    ]
    test_bundle = predictions[
        (dataset_id, representation, algorithm, "test")
    ]

    calibration_frame = calibration_bundle["frame"]
    test_frame = test_bundle["frame"]

    group_overlap = set(
        calibration_frame["duplicate_group_id"]
    ).intersection(
        set(test_frame["duplicate_group_id"])
    )
    record_overlap = set(
        calibration_frame["record_id"]
    ).intersection(
        set(test_frame["record_id"])
    )

    same_class_order = (
        calibration_bundle["class_order"]
        == test_bundle["class_order"]
    )

    prediction_audit_rows.append(
        {
            "dataset_id": dataset_id,
            "representation": representation,
            "algorithm": algorithm,
            "calibration_rows": len(calibration_frame),
            "test_rows": len(test_frame),
            "calibration_groups": (
                calibration_frame["duplicate_group_id"].nunique()
            ),
            "test_groups": (
                test_frame["duplicate_group_id"].nunique()
            ),
            "group_overlap": len(group_overlap),
            "record_overlap": len(record_overlap),
            "same_class_order": same_class_order,
            "calibration_classes": (
                calibration_frame["class_label"].nunique()
            ),
            "test_classes": test_frame["class_label"].nunique(),
        }
    )

prediction_audit = pd.DataFrame(prediction_audit_rows)

if (
    (prediction_audit["group_overlap"] > 0).any()
    or (prediction_audit["record_overlap"] > 0).any()
    or (~prediction_audit["same_class_order"]).any()
):
    display(prediction_audit)
    raise RuntimeError(
        "Falló la auditoría de calibración y prueba."
    )

prediction_audit.to_csv(
    RESULTS_DIR / "01_prediction_input_audit.csv",
    index=False,
    encoding="utf-8-sig",
)

display(prediction_audit)

## Puntuaciones de certeza

Todas se orientan de modo que un valor más alto represente mayor certeza:

- `max_probability`: mayor probabilidad asignada.
- `top2_margin`: diferencia entre la primera y segunda probabilidad.
- `one_minus_normalized_entropy`: uno menos la entropía dividida por \(\log(K)\).

In [ ]:
# ============================================================
# 8. Funciones de incertidumbre y métricas selectivas
# ============================================================
def probability_matrix(bundle, status):
    if status == "native":
        columns = bundle["native_probability_columns"]
    elif status == "temperature_scaled":
        columns = bundle["calibrated_probability_columns"]
    else:
        raise ValueError(status)

    return bundle["frame"][columns].to_numpy(dtype=float)

def calculate_certainty_scores(probabilities):
    probabilities = np.asarray(probabilities, dtype=float)
    n_classes = probabilities.shape[1]

    sorted_probabilities = np.sort(
        probabilities,
        axis=1,
    )
    maximum_probability = sorted_probabilities[:, -1]
    top2_margin = (
        sorted_probabilities[:, -1]
        - sorted_probabilities[:, -2]
    )

    clipped = np.clip(probabilities, 1e-15, 1.0)
    entropy = -np.sum(clipped * np.log(clipped), axis=1)
    normalized_entropy = entropy / np.log(n_classes)
    entropy_certainty = 1.0 - normalized_entropy

    return {
        "max_probability": maximum_probability,
        "top2_margin": top2_margin,
        "one_minus_normalized_entropy": entropy_certainty,
    }

def threshold_for_target_coverage(scores, target_coverage):
    scores = np.asarray(scores, dtype=float)

    if target_coverage >= 1.0:
        return -np.inf

    retained_target = int(
        np.ceil(target_coverage * len(scores))
    )
    retained_target = max(1, min(retained_target, len(scores)))

    ordered = np.sort(scores)[::-1]
    return float(ordered[retained_target - 1])

def encode_labels(labels, class_order):
    mapping = {
        class_label: index
        for index, class_label in enumerate(class_order)
    }
    return np.array(
        [mapping[str(label)] for label in labels],
        dtype=np.int64,
    )

def selective_metrics(
    frame,
    probabilities,
    class_order,
    retain_mask,
):
    retain_mask = np.asarray(retain_mask, dtype=bool)
    n_total = len(frame)
    n_retained = int(retain_mask.sum())

    output = {
        "n_total": n_total,
        "n_retained": n_retained,
        "coverage": float(n_retained / n_total),
    }

    if n_retained == 0:
        output.update(
            {
                "selective_accuracy": np.nan,
                "selective_risk": np.nan,
                "macro_f1": np.nan,
                "balanced_accuracy": np.nan,
                "mcc": np.nan,
                "classes_with_retained_samples": 0,
                "min_class_coverage": 0.0,
            }
        )
        return output

    retained_frame = frame.loc[retain_mask]
    retained_probabilities = probabilities[retain_mask]

    y_true = retained_frame["class_label"].astype(str).to_numpy()
    predicted_indices = retained_probabilities.argmax(axis=1)
    y_pred = np.asarray(class_order, dtype=str)[predicted_indices]

    selective_accuracy = float(
        accuracy_score(y_true, y_pred)
    )
    macro_f1 = float(
        f1_score(
            y_true,
            y_pred,
            labels=class_order,
            average="macro",
            zero_division=0,
        )
    )

    class_recalls = []
    class_coverages = []

    full_labels = frame["class_label"].astype(str).to_numpy()
    for class_label in class_order:
        full_class_mask = full_labels == class_label
        retained_class_mask = y_true == class_label
        class_support = int(full_class_mask.sum())
        retained_support = int(retained_class_mask.sum())

        class_coverages.append(
            retained_support / class_support
            if class_support > 0
            else np.nan
        )

        if retained_support > 0:
            class_recalls.append(
                float(
                    np.mean(
                        y_pred[retained_class_mask]
                        == class_label
                    )
                )
            )

    balanced_accuracy = (
        float(np.mean(class_recalls))
        if class_recalls
        else np.nan
    )

    mcc = (
        float(matthews_corrcoef(y_true, y_pred))
        if len(np.unique(y_true)) > 1
        and len(np.unique(y_pred)) > 1
        else np.nan
    )

    output.update(
        {
            "selective_accuracy": selective_accuracy,
            "selective_risk": 1.0 - selective_accuracy,
            "macro_f1": macro_f1,
            "balanced_accuracy": balanced_accuracy,
            "mcc": mcc,
            "classes_with_retained_samples": len(class_recalls),
            "min_class_coverage": float(
                np.nanmin(class_coverages)
            ),
        }
    )
    return output

def per_class_selective_metrics(
    frame,
    probabilities,
    class_order,
    retain_mask,
):
    retain_mask = np.asarray(retain_mask, dtype=bool)
    predicted_indices = probabilities.argmax(axis=1)
    predicted_labels = np.asarray(
        class_order,
        dtype=str,
    )[predicted_indices]

    true_labels = frame["class_label"].astype(str).to_numpy()
    rows = []

    for class_label in class_order:
        class_mask = true_labels == class_label
        retained_class_mask = class_mask & retain_mask

        support = int(class_mask.sum())
        retained = int(retained_class_mask.sum())
        correct_retained = int(
            np.sum(
                predicted_labels[retained_class_mask]
                == class_label
            )
        )

        class_accuracy = (
            correct_retained / retained
            if retained > 0
            else np.nan
        )

        rows.append(
            {
                "class_label": class_label,
                "support": support,
                "retained": retained,
                "class_coverage": (
                    retained / support if support > 0 else np.nan
                ),
                "correct_retained": correct_retained,
                "class_selective_accuracy": class_accuracy,
                "class_selective_risk": (
                    1.0 - class_accuracy
                    if np.isfinite(class_accuracy)
                    else np.nan
                ),
            }
        )

    return pd.DataFrame(rows)

def risk_coverage_curve(scores, errors):
    scores = np.asarray(scores, dtype=float)
    errors = np.asarray(errors, dtype=float)

    order = np.argsort(-scores, kind="mergesort")
    ordered_errors = errors[order]

    retained_counts = np.arange(
        1,
        len(errors) + 1,
        dtype=float,
    )
    coverage = retained_counts / len(errors)
    risk = np.cumsum(ordered_errors) / retained_counts
    accuracy = 1.0 - risk

    return pd.DataFrame(
        {
            "coverage": coverage,
            "selective_risk": risk,
            "selective_accuracy": accuracy,
        }
    )

def discrete_aurc(scores, errors):
    curve = risk_coverage_curve(scores, errors)
    return float(curve["selective_risk"].mean())

def percentile_interval(values, confidence_level=0.95):
    values = np.asarray(values, dtype=float)
    values = values[np.isfinite(values)]

    lower_probability = (1.0 - confidence_level) / 2.0
    upper_probability = 1.0 - lower_probability

    return (
        float(np.quantile(values, lower_probability)),
        float(np.quantile(values, upper_probability)),
    )

def bootstrap_two_sided_pvalue(delta_values):
    delta_values = np.asarray(delta_values, dtype=float)
    delta_values = delta_values[np.isfinite(delta_values)]
    n = len(delta_values)

    probability_nonpositive = (
        np.count_nonzero(delta_values <= 0) + 1
    ) / (n + 1)
    probability_nonnegative = (
        np.count_nonzero(delta_values >= 0) + 1
    ) / (n + 1)

    return float(
        min(
            1.0,
            2.0
            * min(
                probability_nonpositive,
                probability_nonnegative,
            ),
        )
    )

In [ ]:
# ============================================================
# 9. Selección del score de certeza usando calibración
# ============================================================
score_selection_rows = []

for _, champion in training_champions.iterrows():
    dataset_id = champion["dataset_id"]
    representation = champion["representation"]
    algorithm = champion["algorithm"]

    calibration_bundle = predictions[
        (dataset_id, representation, algorithm, "calibration")
    ]
    calibration_frame = calibration_bundle["frame"]
    class_order = calibration_bundle["class_order"]

    for status in PROBABILITY_STATUSES:
        probabilities = probability_matrix(
            calibration_bundle,
            status,
        )

        scores = calculate_certainty_scores(probabilities)

        true_labels = (
            calibration_frame["class_label"]
            .astype(str)
            .to_numpy()
        )

        predicted_labels = np.asarray(
            class_order,
            dtype=str,
        )[probabilities.argmax(axis=1)]

        errors = (
            predicted_labels != true_labels
        ).astype(float)

        for score_name, score_values in scores.items():
            score_selection_rows.append(
                {
                    "dataset_id": dataset_id,
                    "representation": representation,
                    "algorithm": algorithm,
                    "probability_status": status,
                    "certainty_score": score_name,
                    "calibration_aurc": discrete_aurc(
                        score_values,
                        errors,
                    ),
                    "calibration_error_rate": float(
                        errors.mean()
                    ),
                    "calibration_rows": len(errors),
                }
            )


# ------------------------------------------------------------
# Crear tabla completa de evaluación de scores
# ------------------------------------------------------------
score_selection = pd.DataFrame(score_selection_rows)

if score_selection.empty:
    raise RuntimeError(
        "No se generaron resultados de selección de scores."
    )


# ------------------------------------------------------------
# Seleccionar el menor AURC dentro de calibración
# ------------------------------------------------------------
selected_scores = (
    score_selection.sort_values(
        [
            "dataset_id",
            "probability_status",
            "calibration_aurc",
            "certainty_score",
        ],
        ascending=[
            True,
            True,
            True,
            True,
        ],
    )
    .groupby(
        [
            "dataset_id",
            "probability_status",
        ],
        as_index=False,
    )
    .head(1)
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# Validar que haya exactamente una selección por conjunto
# y estado probabilístico
# ------------------------------------------------------------
expected_selected_scores = (
    len(DATASETS)
    * len(PROBABILITY_STATUSES)
)

if len(selected_scores) != expected_selected_scores:
    display(selected_scores)

    raise RuntimeError(
        "Número inesperado de puntuaciones seleccionadas. "
        f"Obtenidas: {len(selected_scores)}. "
        f"Esperadas: {expected_selected_scores}."
    )


duplicated_selection_mask = selected_scores.duplicated(
    subset=[
        "dataset_id",
        "probability_status",
    ],
    keep=False,
)

if duplicated_selection_mask.any():
    display(
        selected_scores[
            duplicated_selection_mask
        ]
    )

    raise RuntimeError(
        "Existe más de una puntuación de certeza "
        "seleccionada para un mismo conjunto y "
        "estado probabilístico."
    )


# ------------------------------------------------------------
# Marcar las filas seleccionadas en la tabla completa
# ------------------------------------------------------------
score_selection["selected_in_calibration"] = False

for _, selected in selected_scores.iterrows():
    selection_mask = (
        (
            score_selection["dataset_id"]
            == selected["dataset_id"]
        )
        & (
            score_selection["probability_status"]
            == selected["probability_status"]
        )
        & (
            score_selection["certainty_score"]
            == selected["certainty_score"]
        )
    )

    score_selection.loc[
        selection_mask,
        "selected_in_calibration",
    ] = True


# ------------------------------------------------------------
# Crear un conjunto inmutable de claves seleccionadas
#
# Esta estructura evita el error:
# "The truth value of a Series is ambiguous"
# ------------------------------------------------------------
SELECTED_SCORE_KEY_COLUMNS = [
    "dataset_id",
    "probability_status",
    "certainty_score",
]

selected_score_keys = set(
    selected_scores[
        SELECTED_SCORE_KEY_COLUMNS
    ].itertuples(
        index=False,
        name=None,
    )
)

if len(selected_score_keys) != expected_selected_scores:
    raise RuntimeError(
        "Las claves de scores seleccionados no son "
        "completas o no son únicas."
    )


# ------------------------------------------------------------
# Guardar resultados
# ------------------------------------------------------------
score_selection.to_csv(
    TABLES_DIR
    / "Table_calibration_score_selection.csv",
    index=False,
    encoding="utf-8-sig",
)

selected_scores.to_csv(
    RESULTS_DIR
    / "02_selected_certainty_scores.csv",
    index=False,
    encoding="utf-8-sig",
)


# ------------------------------------------------------------
# Mostrar resultados
# ------------------------------------------------------------
print(
    "Número de scores evaluados:",
    len(score_selection),
)

print(
    "Número de scores seleccionados:",
    len(selected_scores),
)

print(
    "Claves seleccionadas:",
    selected_score_keys,
)

display(
    score_selection.sort_values(
        [
            "dataset_id",
            "probability_status",
            "calibration_aurc",
        ]
    )
)

display(selected_scores)

In [ ]:
# ============================================================
# 10. Umbrales fijados en calibración y aplicados a prueba
# ============================================================
operating_point_rows = []
class_operating_point_rows = []


for _, champion in training_champions.iterrows():
    dataset_id = champion["dataset_id"]
    representation = champion["representation"]
    algorithm = champion["algorithm"]

    calibration_key = (
        dataset_id,
        representation,
        algorithm,
        "calibration",
    )

    test_key = (
        dataset_id,
        representation,
        algorithm,
        "test",
    )

    if calibration_key not in predictions:
        raise KeyError(
            "No se encontraron predicciones de calibración "
            f"para {calibration_key}."
        )

    if test_key not in predictions:
        raise KeyError(
            "No se encontraron predicciones de prueba "
            f"para {test_key}."
        )

    calibration_bundle = predictions[
        calibration_key
    ]

    test_bundle = predictions[
        test_key
    ]

    for status in PROBABILITY_STATUSES:
        calibration_probabilities = probability_matrix(
            calibration_bundle,
            status,
        )

        test_probabilities = probability_matrix(
            test_bundle,
            status,
        )

        calibration_scores = calculate_certainty_scores(
            calibration_probabilities
        )

        test_scores = calculate_certainty_scores(
            test_probabilities
        )

        missing_calibration_scores = (
            set(CERTAINTY_SCORES)
            - set(calibration_scores)
        )

        missing_test_scores = (
            set(CERTAINTY_SCORES)
            - set(test_scores)
        )

        if missing_calibration_scores:
            raise KeyError(
                "Faltan scores de calibración: "
                f"{sorted(missing_calibration_scores)}"
            )

        if missing_test_scores:
            raise KeyError(
                "Faltan scores de prueba: "
                f"{sorted(missing_test_scores)}"
            )

        for score_name in CERTAINTY_SCORES:
            calibration_score_values = np.asarray(
                calibration_scores[score_name],
                dtype=float,
            )

            test_score_values = np.asarray(
                test_scores[score_name],
                dtype=float,
            )

            if not np.isfinite(
                calibration_score_values
            ).all():
                raise ValueError(
                    "Se detectaron scores no finitos "
                    "en calibración para "
                    f"{dataset_id}, {status}, {score_name}."
                )

            if not np.isfinite(
                test_score_values
            ).all():
                raise ValueError(
                    "Se detectaron scores no finitos "
                    "en prueba para "
                    f"{dataset_id}, {status}, {score_name}."
                )

            for target_coverage in TARGET_COVERAGES:
                threshold = threshold_for_target_coverage(
                    calibration_score_values,
                    target_coverage,
                )

                calibration_mask = (
                    calibration_score_values
                    >= threshold
                )

                test_mask = (
                    test_score_values
                    >= threshold
                )

                calibration_metrics = selective_metrics(
                    calibration_bundle["frame"],
                    calibration_probabilities,
                    calibration_bundle["class_order"],
                    calibration_mask,
                )

                test_metrics = selective_metrics(
                    test_bundle["frame"],
                    test_probabilities,
                    test_bundle["class_order"],
                    test_mask,
                )

                # ------------------------------------------------
                # Comprobación segura de pertenencia.
                #
                # No se evalúan Series de pandas mediante bool().
                # ------------------------------------------------
                selected_score = (
                    dataset_id,
                    status,
                    score_name,
                ) in selected_score_keys

                row = {
                    "dataset_id": dataset_id,
                    "representation": representation,
                    "algorithm": algorithm,
                    "probability_status": status,
                    "certainty_score": score_name,
                    "selected_in_calibration": bool(
                        selected_score
                    ),
                    "target_calibration_coverage": float(
                        target_coverage
                    ),
                    "certainty_threshold": float(
                        threshold
                    ),
                    "calibration_achieved_coverage": float(
                        calibration_metrics["coverage"]
                    ),
                }

                row.update(
                    {
                        f"test_{metric_name}": metric_value
                        for metric_name, metric_value
                        in test_metrics.items()
                    }
                )

                operating_point_rows.append(row)

                class_metrics = (
                    per_class_selective_metrics(
                        test_bundle["frame"],
                        test_probabilities,
                        test_bundle["class_order"],
                        test_mask,
                    )
                )

                class_metrics.insert(
                    0,
                    "target_calibration_coverage",
                    float(target_coverage),
                )

                class_metrics.insert(
                    0,
                    "selected_in_calibration",
                    bool(selected_score),
                )

                class_metrics.insert(
                    0,
                    "certainty_score",
                    score_name,
                )

                class_metrics.insert(
                    0,
                    "probability_status",
                    status,
                )

                class_metrics.insert(
                    0,
                    "algorithm",
                    algorithm,
                )

                class_metrics.insert(
                    0,
                    "representation",
                    representation,
                )

                class_metrics.insert(
                    0,
                    "dataset_id",
                    dataset_id,
                )

                class_operating_point_rows.append(
                    class_metrics
                )


# ------------------------------------------------------------
# Construcción y validación de tablas
# ------------------------------------------------------------
if not operating_point_rows:
    raise RuntimeError(
        "No se generaron puntos operativos."
    )

if not class_operating_point_rows:
    raise RuntimeError(
        "No se generaron métricas por clase."
    )

operating_points = pd.DataFrame(
    operating_point_rows
)

class_operating_points = pd.concat(
    class_operating_point_rows,
    ignore_index=True,
)


expected_operating_points = (
    len(DATASETS)
    * len(PROBABILITY_STATUSES)
    * len(CERTAINTY_SCORES)
    * len(TARGET_COVERAGES)
)

if len(operating_points) != expected_operating_points:
    raise RuntimeError(
        "Número inesperado de puntos operativos. "
        f"Obtenidos: {len(operating_points)}. "
        f"Esperados: {expected_operating_points}."
    )


selected_operating_points = operating_points[
    operating_points[
        "selected_in_calibration"
    ].astype(bool)
].copy()

expected_selected_operating_points = (
    len(DATASETS)
    * len(PROBABILITY_STATUSES)
    * len(TARGET_COVERAGES)
)

if (
    len(selected_operating_points)
    != expected_selected_operating_points
):
    display(selected_operating_points)

    raise RuntimeError(
        "Número inesperado de puntos operativos "
        "correspondientes a los scores seleccionados. "
        f"Obtenidos: {len(selected_operating_points)}. "
        f"Esperados: "
        f"{expected_selected_operating_points}."
    )


# ------------------------------------------------------------
# Guardar resultados
# ------------------------------------------------------------
operating_points.to_csv(
    TABLES_DIR
    / "Table_calibration_fixed_operating_points.csv",
    index=False,
    encoding="utf-8-sig",
)

class_operating_points.to_csv(
    TABLES_DIR
    / "Table_classwise_selective_performance.csv",
    index=False,
    encoding="utf-8-sig",
)


# ------------------------------------------------------------
# Mostrar resultados seleccionados
# ------------------------------------------------------------
print(
    "Puntos operativos totales:",
    len(operating_points),
)

print(
    "Puntos operativos de scores seleccionados:",
    len(selected_operating_points),
)

display(
    selected_operating_points.sort_values(
        [
            "dataset_id",
            "probability_status",
            "target_calibration_coverage",
        ],
        ascending=[
            True,
            True,
            False,
        ],
    )
)

## Bootstrap agrupado

Los umbrales permanecen fijos durante el bootstrap porque ya fueron establecidos en calibración. Solo se remuestrea el conjunto de prueba, respetando grupos completos dentro de cada clase.

In [ ]:
# ============================================================
# 11. Índices bootstrap de prueba
# ============================================================
def build_group_bootstrap_sampler(base_frame):
    group_to_indices = {
        str(group_id): indices.to_numpy(dtype=int)
        for group_id, indices in base_frame.groupby(
            "duplicate_group_id",
            sort=False,
        ).groups.items()
    }

    group_metadata = (
        base_frame.groupby("duplicate_group_id", sort=False)
        .agg(
            class_label=("class_label", "first"),
            class_count=("class_label", "nunique"),
        )
        .reset_index()
    )
    group_metadata["duplicate_group_id"] = (
        group_metadata["duplicate_group_id"].astype(str)
    )

    if (group_metadata["class_count"] != 1).any():
        raise ValueError("Un grupo contiene más de una clase.")

    groups_by_class = {
        class_label: subset["duplicate_group_id"].to_numpy()
        for class_label, subset in group_metadata.groupby(
            "class_label",
            sort=True,
        )
    }

    def sample_indices(rng):
        sampled_parts = []

        for class_label in sorted(groups_by_class):
            group_ids = groups_by_class[class_label]
            sampled_groups = rng.choice(
                group_ids,
                size=len(group_ids),
                replace=True,
            )
            for group_id in sampled_groups:
                sampled_parts.append(
                    group_to_indices[str(group_id)]
                )

        return np.concatenate(sampled_parts)

    return sample_indices

rng = np.random.default_rng(RANDOM_SEED)
bootstrap_indices = {}

for _, champion in training_champions.iterrows():
    dataset_id = champion["dataset_id"]
    representation = champion["representation"]
    algorithm = champion["algorithm"]

    test_frame = predictions[
        (dataset_id, representation, algorithm, "test")
    ]["frame"]

    sampler = build_group_bootstrap_sampler(test_frame)
    bootstrap_indices[dataset_id] = [
        sampler(rng)
        for _ in range(N_BOOTSTRAP)
    ]

    print(
        dataset_id,
        "| réplicas:",
        len(bootstrap_indices[dataset_id]),
    )

In [ ]:
# ============================================================
# 12. Intervalos de confianza de puntos operativos
#     VERSIÓN OPTIMIZADA
# ============================================================

BOOTSTRAP_METRICS = [
    "coverage",
    "selective_accuracy",
    "selective_risk",
    "macro_f1",
    "balanced_accuracy",
    "mcc",
    "min_class_coverage",
]


# ------------------------------------------------------------
# Función rápida basada únicamente en NumPy
# ------------------------------------------------------------
def fast_selective_metrics(
    y_true_indices,
    predicted_indices,
    certainty_values,
    threshold,
    sampled_indices,
    n_classes,
):
    sampled_indices = np.asarray(
        sampled_indices,
        dtype=np.int64,
    )

    y_sample = y_true_indices[sampled_indices]
    pred_sample = predicted_indices[sampled_indices]
    score_sample = certainty_values[sampled_indices]

    retain_mask = score_sample >= threshold

    n_total = len(sampled_indices)
    n_retained = int(retain_mask.sum())

    coverage = (
        n_retained / n_total
        if n_total > 0
        else np.nan
    )

    # Conteo total por clase antes del rechazo
    total_class_counts = np.bincount(
        y_sample,
        minlength=n_classes,
    ).astype(float)

    if n_retained == 0:
        return {
            "coverage": coverage,
            "selective_accuracy": np.nan,
            "selective_risk": np.nan,
            "macro_f1": np.nan,
            "balanced_accuracy": np.nan,
            "mcc": np.nan,
            "min_class_coverage": 0.0,
        }

    y_retained = y_sample[retain_mask]
    pred_retained = pred_sample[retain_mask]

    confusion_matrix = np.bincount(
        y_retained * n_classes
        + pred_retained,
        minlength=n_classes * n_classes,
    ).reshape(
        n_classes,
        n_classes,
    ).astype(float)

    true_counts = confusion_matrix.sum(axis=1)
    predicted_counts = confusion_matrix.sum(axis=0)
    true_positive = np.diag(confusion_matrix)

    retained_total = confusion_matrix.sum()

    # Exactitud y riesgo
    selective_accuracy = float(
        true_positive.sum() / retained_total
    )

    selective_risk = (
        1.0 - selective_accuracy
    )

    # Macro-F1
    f1_denominator = (
        true_counts + predicted_counts
    )

    class_f1 = np.divide(
        2.0 * true_positive,
        f1_denominator,
        out=np.zeros_like(true_positive),
        where=f1_denominator > 0,
    )

    macro_f1 = float(
        class_f1.mean()
    )

    # Balanced accuracy solo entre clases retenidas
    valid_recall_classes = true_counts > 0

    class_recall = np.divide(
        true_positive,
        true_counts,
        out=np.zeros_like(true_positive),
        where=valid_recall_classes,
    )

    balanced_accuracy = (
        float(
            class_recall[
                valid_recall_classes
            ].mean()
        )
        if valid_recall_classes.any()
        else np.nan
    )

    # MCC multiclase
    correct_total = true_positive.sum()

    numerator = (
        correct_total * retained_total
        - np.dot(
            true_counts,
            predicted_counts,
        )
    )

    denominator_left = (
        retained_total**2
        - np.dot(
            predicted_counts,
            predicted_counts,
        )
    )

    denominator_right = (
        retained_total**2
        - np.dot(
            true_counts,
            true_counts,
        )
    )

    denominator = np.sqrt(
        max(0.0, denominator_left)
        * max(0.0, denominator_right)
    )

    mcc = (
        float(numerator / denominator)
        if denominator > 0
        else 0.0
    )

    # Cobertura mínima entre clases
    retained_class_counts = np.bincount(
        y_retained,
        minlength=n_classes,
    ).astype(float)

    class_coverages = np.divide(
        retained_class_counts,
        total_class_counts,
        out=np.zeros_like(
            retained_class_counts
        ),
        where=total_class_counts > 0,
    )

    valid_classes = total_class_counts > 0

    min_class_coverage = (
        float(
            class_coverages[
                valid_classes
            ].min()
        )
        if valid_classes.any()
        else np.nan
    )

    return {
        "coverage": float(coverage),
        "selective_accuracy": selective_accuracy,
        "selective_risk": selective_risk,
        "macro_f1": macro_f1,
        "balanced_accuracy": balanced_accuracy,
        "mcc": mcc,
        "min_class_coverage": min_class_coverage,
    }


# ------------------------------------------------------------
# Analizar solo scores seleccionados en calibración
# ------------------------------------------------------------
selected_operating_points = operating_points[
    operating_points[
        "selected_in_calibration"
    ].astype(bool)
].copy()

expected_selected_points = (
    len(DATASETS)
    * len(PROBABILITY_STATUSES)
    * len(TARGET_COVERAGES)
)

if (
    len(selected_operating_points)
    != expected_selected_points
):
    display(selected_operating_points)

    raise RuntimeError(
        "Número inesperado de puntos operativos "
        "seleccionados. "
        f"Obtenidos: "
        f"{len(selected_operating_points)}. "
        f"Esperados: "
        f"{expected_selected_points}."
    )


operating_interval_rows = []

print(
    "Puntos operativos que serán analizados:",
    len(selected_operating_points),
)

print(
    "Réplicas bootstrap por punto:",
    N_BOOTSTRAP,
)


# ------------------------------------------------------------
# Recorrer los puntos operativos seleccionados
# ------------------------------------------------------------
for point_number, (
    _,
    operating_point,
) in enumerate(
    selected_operating_points.iterrows(),
    start=1,
):
    dataset_id = operating_point[
        "dataset_id"
    ]

    representation = operating_point[
        "representation"
    ]

    algorithm = operating_point[
        "algorithm"
    ]

    status = operating_point[
        "probability_status"
    ]

    score_name = operating_point[
        "certainty_score"
    ]

    target_coverage = float(
        operating_point[
            "target_calibration_coverage"
        ]
    )

    threshold = float(
        operating_point[
            "certainty_threshold"
        ]
    )

    test_key = (
        dataset_id,
        representation,
        algorithm,
        "test",
    )

    if test_key not in predictions:
        raise KeyError(
            f"No existen predicciones para {test_key}."
        )

    test_bundle = predictions[test_key]
    test_frame = test_bundle["frame"]
    class_order = test_bundle["class_order"]

    probabilities = probability_matrix(
        test_bundle,
        status,
    )

    certainty_scores = (
        calculate_certainty_scores(
            probabilities
        )[score_name]
    )

    class_to_index = {
        class_label: index
        for index, class_label
        in enumerate(class_order)
    }

    y_true_indices = np.array(
        [
            class_to_index[label]
            for label in (
                test_frame["class_label"]
                .astype(str)
                .to_numpy()
            )
        ],
        dtype=np.int64,
    )

    predicted_indices = probabilities.argmax(
        axis=1
    ).astype(np.int64)

    n_classes = len(class_order)

    full_indices = np.arange(
        len(test_frame),
        dtype=np.int64,
    )

    # Métricas puntuales
    point_metrics = fast_selective_metrics(
        y_true_indices=y_true_indices,
        predicted_indices=predicted_indices,
        certainty_values=certainty_scores,
        threshold=threshold,
        sampled_indices=full_indices,
        n_classes=n_classes,
    )

    baseline_metrics = (
        fast_selective_metrics(
            y_true_indices=y_true_indices,
            predicted_indices=predicted_indices,
            certainty_values=np.ones(
                len(test_frame),
                dtype=float,
            ),
            threshold=-np.inf,
            sampled_indices=full_indices,
            n_classes=n_classes,
        )
    )

    bootstrap_values = {
        metric: np.empty(
            N_BOOTSTRAP,
            dtype=float,
        )
        for metric in BOOTSTRAP_METRICS
    }

    risk_reduction_values = np.empty(
        N_BOOTSTRAP,
        dtype=float,
    )

    dataset_bootstrap_indices = (
        bootstrap_indices[dataset_id]
    )

    if (
        len(dataset_bootstrap_indices)
        != N_BOOTSTRAP
    ):
        raise RuntimeError(
            "Número incorrecto de réplicas "
            f"para {dataset_id}."
        )

    # --------------------------------------------------------
    # Bootstrap
    # --------------------------------------------------------
    for bootstrap_index, sampled_indices in enumerate(
        dataset_bootstrap_indices
    ):
        sampled_metrics = fast_selective_metrics(
            y_true_indices=y_true_indices,
            predicted_indices=predicted_indices,
            certainty_values=certainty_scores,
            threshold=threshold,
            sampled_indices=sampled_indices,
            n_classes=n_classes,
        )

        sampled_baseline = fast_selective_metrics(
            y_true_indices=y_true_indices,
            predicted_indices=predicted_indices,
            certainty_values=np.ones(
                len(test_frame),
                dtype=float,
            ),
            threshold=-np.inf,
            sampled_indices=sampled_indices,
            n_classes=n_classes,
        )

        for metric in BOOTSTRAP_METRICS:
            bootstrap_values[
                metric
            ][bootstrap_index] = (
                sampled_metrics[metric]
            )

        risk_reduction_values[
            bootstrap_index
        ] = (
            sampled_baseline[
                "selective_risk"
            ]
            - sampled_metrics[
                "selective_risk"
            ]
        )

    # --------------------------------------------------------
    # Intervalos de métricas
    # --------------------------------------------------------
    for metric in BOOTSTRAP_METRICS:
        metric_values = bootstrap_values[
            metric
        ]

        finite_values = metric_values[
            np.isfinite(metric_values)
        ]

        if len(finite_values) == 0:
            ci_low = np.nan
            ci_high = np.nan
        else:
            ci_low, ci_high = (
                percentile_interval(
                    finite_values,
                    CONFIDENCE_LEVEL,
                )
            )

        operating_interval_rows.append(
            {
                "dataset_id": dataset_id,
                "representation": representation,
                "algorithm": algorithm,
                "probability_status": status,
                "certainty_score": score_name,
                "selected_in_calibration": True,
                "target_calibration_coverage": (
                    target_coverage
                ),
                "certainty_threshold": threshold,
                "metric": metric,
                "estimate": point_metrics[metric],
                "ci_low": ci_low,
                "ci_high": ci_high,
                "confidence_level": (
                    CONFIDENCE_LEVEL
                ),
                "n_bootstrap": N_BOOTSTRAP,
            }
        )

    # --------------------------------------------------------
    # Reducción del riesgo frente a no rechazar
    # --------------------------------------------------------
    finite_risk_reductions = (
        risk_reduction_values[
            np.isfinite(
                risk_reduction_values
            )
        ]
    )

    risk_reduction_ci_low, (
        risk_reduction_ci_high
    ) = percentile_interval(
        finite_risk_reductions,
        CONFIDENCE_LEVEL,
    )

    operating_interval_rows.append(
        {
            "dataset_id": dataset_id,
            "representation": representation,
            "algorithm": algorithm,
            "probability_status": status,
            "certainty_score": score_name,
            "selected_in_calibration": True,
            "target_calibration_coverage": (
                target_coverage
            ),
            "certainty_threshold": threshold,
            "metric": (
                "risk_reduction_vs_no_rejection"
            ),
            "estimate": (
                baseline_metrics[
                    "selective_risk"
                ]
                - point_metrics[
                    "selective_risk"
                ]
            ),
            "ci_low": risk_reduction_ci_low,
            "ci_high": risk_reduction_ci_high,
            "confidence_level": (
                CONFIDENCE_LEVEL
            ),
            "n_bootstrap": N_BOOTSTRAP,
        }
    )

    print(
        f"[{point_number}/"
        f"{len(selected_operating_points)}] "
        f"{dataset_id} | {status} | "
        f"{score_name} | "
        f"cobertura objetivo="
        f"{target_coverage:.2f} completado"
    )


# ------------------------------------------------------------
# Crear y validar tabla final
# ------------------------------------------------------------
operating_point_intervals = pd.DataFrame(
    operating_interval_rows
)

expected_interval_rows = (
    len(selected_operating_points)
    * (
        len(BOOTSTRAP_METRICS)
        + 1
    )
)

if (
    len(operating_point_intervals)
    != expected_interval_rows
):
    raise RuntimeError(
        "Número inesperado de intervalos. "
        f"Obtenidos: "
        f"{len(operating_point_intervals)}. "
        f"Esperados: "
        f"{expected_interval_rows}."
    )


# ------------------------------------------------------------
# Guardar
# ------------------------------------------------------------
operating_point_intervals.to_csv(
    TABLES_DIR
    / "Table_selective_operating_point_intervals.csv",
    index=False,
    encoding="utf-8-sig",
)


# ------------------------------------------------------------
# Mostrar resultados principales
# ------------------------------------------------------------
print(
    "Celda 12 completada correctamente."
)

print(
    "Filas generadas:",
    len(operating_point_intervals),
)

display(
    operating_point_intervals[
        (
            operating_point_intervals[
                "probability_status"
            ]
            == PRIMARY_PROBABILITY_STATUS
        )
        & (
            operating_point_intervals[
                "selected_in_calibration"
            ].astype(bool)
        )
        & (
            operating_point_intervals[
                "metric"
            ].isin(
                [
                    "coverage",
                    "selective_accuracy",
                    "selective_risk",
                    (
                        "risk_reduction_"
                        "vs_no_rejection"
                    ),
                ]
            )
        )
    ].sort_values(
        [
            "dataset_id",
            "target_calibration_coverage",
            "metric",
        ],
        ascending=[
            True,
            False,
            True,
        ],
    )
)

In [ ]:
# ============================================================
# 13. Curvas riesgo–cobertura y AURC en prueba
# ============================================================
risk_curve_rows = []
aurc_rows = []
aurc_bootstrap_values = {}


for _, champion in training_champions.iterrows():
    dataset_id = champion["dataset_id"]
    representation = champion["representation"]
    algorithm = champion["algorithm"]

    test_key = (
        dataset_id,
        representation,
        algorithm,
        "test",
    )

    if test_key not in predictions:
        raise KeyError(
            "No se encontraron predicciones de prueba "
            f"para {test_key}."
        )

    test_bundle = predictions[
        test_key
    ]

    test_frame = test_bundle["frame"]
    class_order = test_bundle["class_order"]

    true_labels = (
        test_frame["class_label"]
        .astype(str)
        .to_numpy()
    )

    if dataset_id not in bootstrap_indices:
        raise KeyError(
            "No existen índices bootstrap para "
            f"{dataset_id}."
        )

    if (
        len(bootstrap_indices[dataset_id])
        != N_BOOTSTRAP
    ):
        raise RuntimeError(
            "Número incompleto de réplicas bootstrap "
            f"para {dataset_id}."
        )

    for status in PROBABILITY_STATUSES:
        probabilities = probability_matrix(
            test_bundle,
            status,
        )

        predicted_labels = np.asarray(
            class_order,
            dtype=str,
        )[probabilities.argmax(axis=1)]

        errors = (
            predicted_labels != true_labels
        ).astype(float)

        scores = calculate_certainty_scores(
            probabilities
        )

        for score_name in CERTAINTY_SCORES:
            if score_name not in scores:
                raise KeyError(
                    f"No existe el score {score_name} "
                    f"para {dataset_id}, {status}."
                )

            score_values = np.asarray(
                scores[score_name],
                dtype=float,
            )

            if not np.isfinite(score_values).all():
                raise ValueError(
                    "Se detectaron scores no finitos para "
                    f"{dataset_id}, {status}, {score_name}."
                )

            curve = risk_coverage_curve(
                score_values,
                errors,
            )

            curve.insert(
                0,
                "certainty_score",
                score_name,
            )

            curve.insert(
                0,
                "probability_status",
                status,
            )

            curve.insert(
                0,
                "algorithm",
                algorithm,
            )

            curve.insert(
                0,
                "representation",
                representation,
            )

            curve.insert(
                0,
                "dataset_id",
                dataset_id,
            )

            risk_curve_rows.append(curve)

            point_aurc = discrete_aurc(
                score_values,
                errors,
            )

            values = np.empty(
                N_BOOTSTRAP,
                dtype=float,
            )

            for (
                bootstrap_index,
                sampled_indices,
            ) in enumerate(
                bootstrap_indices[dataset_id]
            ):
                sampled_indices = np.asarray(
                    sampled_indices,
                    dtype=int,
                )

                values[
                    bootstrap_index
                ] = discrete_aurc(
                    score_values[
                        sampled_indices
                    ],
                    errors[
                        sampled_indices
                    ],
                )

            aurc_bootstrap_values[
                (
                    dataset_id,
                    status,
                    score_name,
                )
            ] = values

            ci_low, ci_high = percentile_interval(
                values,
                CONFIDENCE_LEVEL,
            )

            # ------------------------------------------------
            # Comprobación segura mediante pertenencia.
            # ------------------------------------------------
            selected_score = (
                dataset_id,
                status,
                score_name,
            ) in selected_score_keys

            aurc_rows.append(
                {
                    "dataset_id": dataset_id,
                    "representation": representation,
                    "algorithm": algorithm,
                    "probability_status": status,
                    "certainty_score": score_name,
                    "selected_in_calibration": bool(
                        selected_score
                    ),
                    "test_aurc": float(
                        point_aurc
                    ),
                    "ci_low": float(
                        ci_low
                    ),
                    "ci_high": float(
                        ci_high
                    ),
                    "confidence_level": float(
                        CONFIDENCE_LEVEL
                    ),
                    "n_bootstrap": int(
                        N_BOOTSTRAP
                    ),
                }
            )


# ------------------------------------------------------------
# Construcción y validación de resultados
# ------------------------------------------------------------
if not risk_curve_rows:
    raise RuntimeError(
        "No se generaron curvas riesgo–cobertura."
    )

if not aurc_rows:
    raise RuntimeError(
        "No se generaron resultados AURC."
    )

risk_coverage_curves = pd.concat(
    risk_curve_rows,
    ignore_index=True,
)

aurc_results = pd.DataFrame(
    aurc_rows
)


expected_aurc_rows = (
    len(DATASETS)
    * len(PROBABILITY_STATUSES)
    * len(CERTAINTY_SCORES)
)

if len(aurc_results) != expected_aurc_rows:
    raise RuntimeError(
        "Número inesperado de resultados AURC. "
        f"Obtenidos: {len(aurc_results)}. "
        f"Esperados: {expected_aurc_rows}."
    )


selected_aurc_results = aurc_results[
    aurc_results[
        "selected_in_calibration"
    ].astype(bool)
].copy()

expected_selected_aurc_rows = (
    len(DATASETS)
    * len(PROBABILITY_STATUSES)
)

if (
    len(selected_aurc_results)
    != expected_selected_aurc_rows
):
    display(selected_aurc_results)

    raise RuntimeError(
        "Número inesperado de resultados AURC "
        "para los scores seleccionados."
    )


# ------------------------------------------------------------
# Guardar resultados
# ------------------------------------------------------------
risk_coverage_curves.to_csv(
    RESULTS_DIR
    / "03_test_risk_coverage_curves.csv",
    index=False,
    encoding="utf-8-sig",
)

aurc_results.to_csv(
    TABLES_DIR
    / "Table_test_aurc_results.csv",
    index=False,
    encoding="utf-8-sig",
)


# ------------------------------------------------------------
# Mostrar resultados
# ------------------------------------------------------------
print(
    "Filas de curvas riesgo–cobertura:",
    len(risk_coverage_curves),
)

print(
    "Resultados AURC:",
    len(aurc_results),
)

display(
    aurc_results.sort_values(
        [
            "dataset_id",
            "probability_status",
            "test_aurc",
        ]
    )
)

print(
    "Resultados AURC correspondientes "
    "a los scores seleccionados:"
)

display(
    selected_aurc_results.sort_values(
        [
            "dataset_id",
            "probability_status",
        ]
    )
)

In [ ]:
# ============================================================
# 14. Efecto de la calibración sobre AURC
# ============================================================
aurc_calibration_rows = []

for dataset_id in DATASETS:
    for score_name in CERTAINTY_SCORES:
        native_values = aurc_bootstrap_values[
            (dataset_id, "native", score_name)
        ]
        calibrated_values = aurc_bootstrap_values[
            (
                dataset_id,
                "temperature_scaled",
                score_name,
            )
        ]
        delta_values = calibrated_values - native_values

        native_point = aurc_results[
            (aurc_results["dataset_id"] == dataset_id)
            & (
                aurc_results["probability_status"]
                == "native"
            )
            & (
                aurc_results["certainty_score"]
                == score_name
            )
        ]["test_aurc"].iloc[0]

        calibrated_point = aurc_results[
            (aurc_results["dataset_id"] == dataset_id)
            & (
                aurc_results["probability_status"]
                == "temperature_scaled"
            )
            & (
                aurc_results["certainty_score"]
                == score_name
            )
        ]["test_aurc"].iloc[0]

        ci_low, ci_high = percentile_interval(
            delta_values,
            CONFIDENCE_LEVEL,
        )

        aurc_calibration_rows.append(
            {
                "dataset_id": dataset_id,
                "certainty_score": score_name,
                "native_aurc": native_point,
                "temperature_scaled_aurc": calibrated_point,
                "delta_scaled_minus_native": (
                    calibrated_point - native_point
                ),
                "ci_low": ci_low,
                "ci_high": ci_high,
                "bootstrap_p_raw": (
                    bootstrap_two_sided_pvalue(delta_values)
                ),
                "improved_if_negative": (
                    calibrated_point < native_point
                ),
            }
        )

aurc_calibration_effects = pd.DataFrame(
    aurc_calibration_rows
)
aurc_calibration_effects["bootstrap_p_holm"] = np.nan
aurc_calibration_effects["reject_holm_0_05"] = False

for dataset_id, row_indices in (
    aurc_calibration_effects.groupby(
        "dataset_id"
    ).groups.items()
):
    raw_p = aurc_calibration_effects.loc[
        row_indices,
        "bootstrap_p_raw",
    ].to_numpy(dtype=float)

    reject, adjusted, _, _ = multipletests(
        raw_p,
        alpha=0.05,
        method="holm",
    )

    aurc_calibration_effects.loc[
        row_indices,
        "bootstrap_p_holm",
    ] = adjusted
    aurc_calibration_effects.loc[
        row_indices,
        "reject_holm_0_05",
    ] = reject

aurc_calibration_effects.to_csv(
    TABLES_DIR / "Table_calibration_effect_on_aurc.csv",
    index=False,
    encoding="utf-8-sig",
)

display(aurc_calibration_effects)

In [ ]:
# ============================================================
# 15. Análisis primario por clase al 90% de cobertura objetivo
# ============================================================
primary_class_rows = []

for dataset_id in DATASETS:
    selected_score = selected_scores[
        (selected_scores["dataset_id"] == dataset_id)
        & (
            selected_scores["probability_status"]
            == PRIMARY_PROBABILITY_STATUS
        )
    ]["certainty_score"].iloc[0]

    subset = class_operating_points[
        (class_operating_points["dataset_id"] == dataset_id)
        & (
            class_operating_points["probability_status"]
            == PRIMARY_PROBABILITY_STATUS
        )
        & (
            class_operating_points["certainty_score"]
            == selected_score
        )
        & (
            class_operating_points[
                "target_calibration_coverage"
            ]
            == PRIMARY_CLASS_ANALYSIS_COVERAGE
        )
    ].copy()

    subset["primary_analysis"] = True
    primary_class_rows.append(subset)

primary_class_analysis = pd.concat(
    primary_class_rows,
    ignore_index=True,
)

primary_class_analysis.to_csv(
    TABLES_DIR / "Table_primary_classwise_90pct_coverage.csv",
    index=False,
    encoding="utf-8-sig",
)

display(primary_class_analysis)

## Figuras

Las curvas completas de prueba son descriptivas. Los puntos operativos utilizados para conclusiones prácticas se fijaron previamente en calibración.

In [ ]:
# ============================================================
# 16. Figuras de riesgo–cobertura
# ============================================================
for dataset_id in DATASETS:
    figure, axis = plt.subplots(figsize=(8, 6))

    subset = risk_coverage_curves[
        (risk_coverage_curves["dataset_id"] == dataset_id)
        & (
            risk_coverage_curves["probability_status"]
            == PRIMARY_PROBABILITY_STATUS
        )
    ]

    for score_name in CERTAINTY_SCORES:
        score_subset = subset[
            subset["certainty_score"] == score_name
        ]
        axis.plot(
            score_subset["coverage"],
            score_subset["selective_risk"],
            label=score_name,
        )

    axis.set_xlabel("Cobertura")
    axis.set_ylabel("Riesgo selectivo")
    axis.set_title(
        f"Curvas riesgo–cobertura — {dataset_id}"
    )
    axis.set_xlim(0.5, 1.0)
    axis.grid(True, alpha=0.3)
    axis.legend()
    figure.tight_layout()

    figure.savefig(
        FIGURES_DIR
        / f"risk_coverage_scores_{dataset_id}.png",
        dpi=300,
        bbox_inches="tight",
    )
    figure.savefig(
        FIGURES_DIR
        / f"risk_coverage_scores_{dataset_id}.pdf",
        bbox_inches="tight",
    )
    plt.show()

In [ ]:
# ============================================================
# 17. Figuras: score seleccionado, nativo frente a calibrado
# ============================================================
for dataset_id in DATASETS:
    selected_score = selected_scores[
        (selected_scores["dataset_id"] == dataset_id)
        & (
            selected_scores["probability_status"]
            == PRIMARY_PROBABILITY_STATUS
        )
    ]["certainty_score"].iloc[0]

    figure, axis = plt.subplots(figsize=(8, 6))

    subset = risk_coverage_curves[
        (risk_coverage_curves["dataset_id"] == dataset_id)
        & (
            risk_coverage_curves["certainty_score"]
            == selected_score
        )
    ]

    for status in PROBABILITY_STATUSES:
        status_subset = subset[
            subset["probability_status"] == status
        ]
        axis.plot(
            status_subset["coverage"],
            status_subset["selective_risk"],
            label=status,
        )

    axis.set_xlabel("Cobertura")
    axis.set_ylabel("Riesgo selectivo")
    axis.set_title(
        f"{dataset_id}: {selected_score}"
    )
    axis.set_xlim(0.5, 1.0)
    axis.grid(True, alpha=0.3)
    axis.legend()
    figure.tight_layout()

    figure.savefig(
        FIGURES_DIR
        / f"risk_coverage_native_vs_scaled_{dataset_id}.png",
        dpi=300,
        bbox_inches="tight",
    )
    figure.savefig(
        FIGURES_DIR
        / f"risk_coverage_native_vs_scaled_{dataset_id}.pdf",
        bbox_inches="tight",
    )
    plt.show()

In [ ]:
# ============================================================
# 18. Figuras: cobertura y exactitud por clase al 90%
# ============================================================
for dataset_id in DATASETS:
    subset = primary_class_analysis[
        primary_class_analysis["dataset_id"] == dataset_id
    ].sort_values("class_label")

    figure, axis = plt.subplots(figsize=(9, 5))
    x = np.arange(len(subset))
    width = 0.38

    axis.bar(
        x - width / 2,
        subset["class_coverage"],
        width,
        label="Cobertura por clase",
    )
    axis.bar(
        x + width / 2,
        subset["class_selective_accuracy"],
        width,
        label="Exactitud selectiva",
    )

    axis.set_xticks(x)
    axis.set_xticklabels(
        subset["class_label"],
        rotation=30,
        ha="right",
    )
    axis.set_ylim(0.0, 1.05)
    axis.set_ylabel("Proporción")
    axis.set_title(
        f"Rendimiento selectivo por clase — {dataset_id}"
    )
    axis.grid(True, axis="y", alpha=0.3)
    axis.legend()
    figure.tight_layout()

    figure.savefig(
        FIGURES_DIR
        / f"classwise_selective_90pct_{dataset_id}.png",
        dpi=300,
        bbox_inches="tight",
    )
    figure.savefig(
        FIGURES_DIR
        / f"classwise_selective_90pct_{dataset_id}.pdf",
        bbox_inches="tight",
    )
    plt.show()

In [ ]:
# ============================================================
# 19. Resumen automático de resultados
# ============================================================
summary_rows = []

primary_intervals = operating_point_intervals[
    (
        operating_point_intervals["probability_status"]
        == PRIMARY_PROBABILITY_STATUS
    )
    & operating_point_intervals["selected_in_calibration"]
    & (
        operating_point_intervals["metric"].isin(
            [
                "coverage",
                "selective_accuracy",
                "selective_risk",
                "risk_reduction_vs_no_rejection",
            ]
        )
    )
].copy()

for _, row in primary_intervals.iterrows():
    summary_rows.append(
        {
            "analysis": "primary_operating_point",
            "dataset_id": row["dataset_id"],
            "certainty_score": row["certainty_score"],
            "target_calibration_coverage": (
                row["target_calibration_coverage"]
            ),
            "metric": row["metric"],
            "estimate": row["estimate"],
            "ci_low": row["ci_low"],
            "ci_high": row["ci_high"],
        }
    )

for _, row in aurc_calibration_effects.iterrows():
    summary_rows.append(
        {
            "analysis": "calibration_effect_on_aurc",
            "dataset_id": row["dataset_id"],
            "certainty_score": row["certainty_score"],
            "target_calibration_coverage": np.nan,
            "metric": "delta_scaled_minus_native_aurc",
            "estimate": row["delta_scaled_minus_native"],
            "ci_low": row["ci_low"],
            "ci_high": row["ci_high"],
        }
    )

automatic_summary = pd.DataFrame(summary_rows)
automatic_summary.to_csv(
    RESULTS_DIR / "04_automatic_selective_summary.csv",
    index=False,
    encoding="utf-8-sig",
)

display(automatic_summary.head(30))

In [ ]:
# ============================================================
# 20. Protocolo, validaciones y manifiesto de integridad
# ============================================================
protocol = {
    "protocol_name": (
        "Grain_AI_MDPI_selective_uncertainty_protocol_v1"
    ),
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "random_seed": RANDOM_SEED,
    "n_bootstrap": N_BOOTSTRAP,
    "confidence_level": CONFIDENCE_LEVEL,
    "primary_probability_status": PRIMARY_PROBABILITY_STATUS,
    "certainty_scores": CERTAINTY_SCORES,
    "target_coverages": TARGET_COVERAGES,
    "score_selection_partition": "calibration",
    "threshold_definition_partition": "calibration",
    "final_evaluation_partition": "test",
    "bootstrap_design": {
        "paired": True,
        "stratified_by_class": True,
        "resampling_unit": "duplicate_group_id",
        "thresholds_fixed_during_bootstrap": True,
    },
    "primary_models": training_champions[
        [
            "dataset_id",
            "representation",
            "algorithm",
            "candidate_id",
        ]
    ].to_dict(orient="records"),
    "restrictions": [
        "No model retraining.",
        "No hyperparameter reselection.",
        "No test-based threshold selection.",
        "No test-based certainty-score selection.",
        "No new champion selected from test.",
        "No image-level robustness claim.",
        "Risk-coverage curves on test are descriptive.",
    ],
}

(PROTOCOL_DIR / "selective_uncertainty_protocol_v1.json").write_text(
    json.dumps(protocol, indent=2, ensure_ascii=False),
    encoding="utf-8",
)

validation_checks = {
    "notebook06_all_checks_passed": bool(
        notebook06_validation["passed"].astype(bool).all()
    ),
    "champions_match_notebooks04_and06": bool(
        champion_crosscheck["matches"].all()
    ),
    "exactly_two_primary_models": (
        len(training_champions) == len(DATASETS)
    ),
    "all_prediction_inputs_loaded": (
        len(predictions) == len(DATASETS) * 2
    ),
    "no_calibration_test_group_overlap": bool(
        (prediction_audit["group_overlap"] == 0).all()
    ),
    "no_calibration_test_record_overlap": bool(
        (prediction_audit["record_overlap"] == 0).all()
    ),
    "certainty_score_selected_on_calibration": (
        len(selected_scores)
        == len(DATASETS) * len(PROBABILITY_STATUSES)
    ),
    "all_operating_thresholds_from_calibration": True,
    "all_bootstrap_replicates_complete": all(
        len(values) == N_BOOTSTRAP
        for values in bootstrap_indices.values()
    ),
    "no_new_model_selected_from_test": True,
    "no_test_threshold_optimization": True,
}

final_validation = pd.DataFrame(
    [
        {"check": key, "passed": value}
        for key, value in validation_checks.items()
    ]
)

final_validation.to_csv(
    RESULTS_DIR / "05_final_selective_validation.csv",
    index=False,
    encoding="utf-8-sig",
)

if not final_validation["passed"].all():
    display(final_validation)
    raise RuntimeError(
        "Falló al menos una validación final."
    )

def sha256_file(path: Path):
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        for block in iter(
            lambda: stream.read(1024 * 1024),
            b"",
        ):
            digest.update(block)
    return digest.hexdigest()

manifest_rows = []

for path in sorted(OUTPUT_ROOT.rglob("*")):
    if path.is_file():
        manifest_rows.append(
            {
                "relative_path": str(
                    path.relative_to(OUTPUT_ROOT)
                ),
                "size_bytes": path.stat().st_size,
                "sha256": sha256_file(path),
            }
        )

integrity_manifest = pd.DataFrame(manifest_rows)
integrity_manifest.to_csv(
    RESULTS_DIR / "06_output_integrity_manifest.csv",
    index=False,
    encoding="utf-8-sig",
)

print("Validación final: CORRECTA")
display(final_validation)
print("Archivos registrados:", len(integrity_manifest))

In [ ]:
# ============================================================
# 21. Exportación a Drive y ZIP final
# ============================================================
DRIVE_NOTEBOOK07_ROOT = (
    DRIVE_PROJECT_ROOT
    / "NOTEBOOK07_SELECTIVE_UNCERTAINTY_OUTPUTS"
)

if DRIVE_NOTEBOOK07_ROOT.exists():
    shutil.rmtree(DRIVE_NOTEBOOK07_ROOT)

shutil.copytree(
    OUTPUT_ROOT,
    DRIVE_NOTEBOOK07_ROOT,
)

FINAL_ZIP_PATH = (
    DRIVE_PROJECT_ROOT
    / "NOTEBOOK07_SELECTIVE_UNCERTAINTY_OUTPUTS.zip"
)

if FINAL_ZIP_PATH.exists():
    FINAL_ZIP_PATH.unlink()

with zipfile.ZipFile(
    FINAL_ZIP_PATH,
    mode="w",
    compression=zipfile.ZIP_DEFLATED,
    compresslevel=6,
) as archive:
    for path in sorted(OUTPUT_ROOT.rglob("*")):
        if path.is_file():
            archive.write(
                path,
                arcname=str(
                    path.relative_to(OUTPUT_ROOT)
                ),
            )

print("Directorio exportado:", DRIVE_NOTEBOOK07_ROOT)
print("ZIP final:", FINAL_ZIP_PATH)
print(
    "Tamaño ZIP (MB):",
    round(FINAL_ZIP_PATH.stat().st_size / 1e6, 2),
)

# Interpretación editorial

Los resultados deben interpretarse con las siguientes reglas:

- Una reducción de riesgo con intervalo completamente superior a cero indica que el rechazo de muestras de baja certeza mejora la seguridad predictiva frente a clasificar todos los granos.
- La cobertura de prueba puede diferir de la cobertura objetivo porque el umbral se fijó en calibración.
- Una cobertura global alta puede ocultar una cobertura baja en una clase específica; por eso se reporta `min_class_coverage`.
- AURC menor indica mejor ordenamiento de las muestras desde las más seguras hasta las más riesgosas.
- La puntuación de certeza principal se selecciona en calibración, no en prueba.
- El análisis con probabilidades nativas funciona como sensibilidad respecto del escalamiento por temperatura.
- La clasificación selectiva no demuestra generalización externa ni robustez ante cambios de adquisición.